# Statistical Tables

Formatted tables of the inferential and descriptive statistics for the
experiment, read from the canonical outputs of the analysis pipeline.

**Invariants**

1. Every number is computed by `python -m maestro.analysis`, which writes JSON
   plus a `report.md` under `output/analysis/<timestamp>/`. This notebook only
   formats those files; it never recomputes a statistic, so a table here can
   never disagree with the pipeline output.
2. The setup cell reads the newest analysis run, and runs the pipeline once if
   none exists yet.
3. Two scoring conventions are reported side by side: `intent_to_treat`
   (every run counts, a failed or unrenderable diagram scores 0) and
   `valid_only` (only renderable diagrams). See the pipeline docs for the
   full rationale.

## Catalogue

| Table | Source file |
|---|---|
| Mean entity-identification F1 per strategy | (query: `queries.py`) |
| One-way ANOVA of F1 by strategy | `anova_strategy__*.json`, `effect_sizes__*.json` |
| Two-way ANOVA of F1 (strategy x tier) | `anova_strategy_by_tier__*.json` |
| Correctness and run outcomes by input complexity | (query: `queries.py`) |
| Mean errors per valid diagram | (query: `queries.py`) |
| Efficiency per run by strategy | (query: `queries.py`) |
| Two-way ANOVA of F1 (strategy x model) | `anova_strategy_by_model__*.json` |
| Mixed-effects estimates for F1 | `mixed_effects_robustness__*.json` |
| Reliability and accuracy per model | (query: `queries.py`) |

## Setup

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

try:
    import maestro  # noqa: F401
except ModuleNotFoundError:
    _p = Path.cwd()
    while not (_p / 'src' / 'maestro').exists() and _p != _p.parent:
        _p = _p.parent
    sys.path.insert(0, str(_p / 'src'))

import sqlite3

import pandas as pd

from maestro.analysis import statistics as S
from maestro.viz import db as viz_db

REPO = Path.cwd()
while not (REPO / 'out' / 'maestro.db').exists() and REPO != REPO.parent:
    REPO = REPO.parent
DB_PATH = REPO / 'out' / 'maestro.db'
ANALYSIS_ROOT = REPO / 'output' / 'analysis'


def _latest_run(root: Path) -> Path | None:
    runs = [p for p in root.glob('*') if p.is_dir() and (p / 'report.md').exists()]
    return max(runs, key=lambda p: p.name) if runs else None


# Read the canonical analysis outputs, never recomputing the statistics here:
# the tables are a formatted view of what `python -m maestro.analysis` emits,
# so a table can never drift from report.md. Run the pipeline once if no
# output exists yet, then read it.
run_dir = _latest_run(ANALYSIS_ROOT)
if run_dir is None:
    print('No analysis output found; running the pipeline once...')
    subprocess.run(
        [sys.executable, '-m', 'maestro.analysis', '--db', str(DB_PATH)],
        cwd=REPO, check=True,
    )
    run_dir = _latest_run(ANALYSIS_ROOT)

if run_dir is None:
    raise RuntimeError('analysis pipeline produced no readable run directory')

A = {p.stem: json.loads(p.read_text()) for p in run_dir.glob('*.json')}

ITT = 'intent_to_treat'
VO = 'valid_only'
STRATEGY_DISPLAY = {
    'single_agent': 'Single Agent', 'sop_based': 'SOP',
    'crew_ai': 'CrewAI', 'lang_graph': 'LangGraph',
}


def strat(value: str) -> str:
    return STRATEGY_DISPLAY.get(value, value)


# Per-cell aggregation for any table whose grain is per-cell means (the
# convention analysis.statistics uses for every inferential number; see the
# module docstring). Loaded lazily and cached, so a tables-only session that
# needs no per-cell frame never opens the DB.
_SDF = None


def per_cell_means(convention):
    global _SDF
    if _SDF is None:
        with viz_db.connect(DB_PATH) as _conn:
            _conn.row_factory = sqlite3.Row
            _SDF = S.load_dataframe(_conn)
    return S.aggregate_experimental(_SDF, convention)


print('reading analysis run:', run_dir.name)
print('schema version:', A.get('descriptive', {}).get('schema_version'))
print('files:', len(A))


# ---- Shared table formatters -------------------------------------------
# Defined once here so every table cell uses one set of rules regardless of
# run order. APA style throughout: a leading zero is dropped from any value
# that cannot exceed 1 (p, partial eta^2), and p < .001 is reported as a
# bound rather than a rounded 2-dp figure that would read as .00.


def p_apa(x):
    # APA style: p < .001 as a bound, otherwise 2 decimals. Fall back to 3
    # decimals when 2 would round to .00 (a value like .004 must not read as
    # zero), the smallest precision that keeps a small-but-nonzero p honest.
    if x < 0.001:
        return '< .001'
    two = f'{x:.2f}'
    return (two if float(two) != 0.0 else f'{x:.3f}').lstrip('0')


def eta_apa(x):
    return f'{x:.3f}'.lstrip('0')


def term_label(term):
    # Patsy term -> readable factor name; interactions joined with a times
    # sign, a single factor stripped of its C(...) / Treatment wrapper.
    parts = ['Strategy' if 'strategy' in t else 'Tier' if 'tier' in t
             else 'Model' if 'model' in t else t
             for t in term.split(':')]
    return ' × '.join(parts)


def anova_terms_table(key):
    an = A[key]
    resid = int(an['residual_df'])
    return pd.DataFrame([
        {'Source': term_label(term),
         'df': f"{int(v['df'])}, {resid}",
         'F': f"{v['F']:.2f}",
         'p': p_apa(v['p']),
         'Partial eta^2': eta_apa(v['partial_eta_sq'])}
        for term, v in an['terms'].items()
    ]).set_index('Source')

## Mean entity-identification F1 per strategy

Per-strategy mean `entity_id_f1` under both scoring conventions. Grand means
over the raw runs, distinct from the per-cell means the inferential tests use
(they differ by at most ~0.001). "Valid diagrams only" averages the runs that
parsed; "Every run" is the intent-to-treat convention, counting failed and
invalid runs as zero.

In [ ]:
from maestro.viz import queries as q

# Display labels for this table: hyphenated forms match the report style.
LABELS = {'single_agent': 'Single-agent', 'sop_based': 'SOP-based',
          'crew_ai': 'CrewAI', 'lang_graph': 'LangGraph'}
ORDER = ['single_agent', 'sop_based', 'crew_ai', 'lang_graph']

with viz_db.connect(DB_PATH) as conn:
    rows = {r[0]: r for r in q.mean_entity_id_f1_by_strategy_by_convention(conn)}

mean_f1 = pd.DataFrame(
    [{'Strategy': LABELS[s],
      'Valid diagrams only': round(rows[s][1], 3),
      'Every run (failures scored 0)': round(rows[s][2], 3)}
     for s in ORDER]
).set_index('Strategy')

mean_f1

## One-way ANOVA of entity-identification F1 by strategy

Omnibus test of the strategy effect on `entity_id_f1`, under both scoring
conventions. Computed on per-cell means (one per strategy x model x input),
using Type II sums of squares. The `|d| range` column gives the range of
absolute Cohen's d across the six pairwise strategy contrasts; the largest
under both conventions is CrewAI against the single-agent baseline. Every
interval and effect size is negligible: the strategies do not separate.

The p and partial-eta-squared columns follow APA style (no leading zero on a
quantity that cannot exceed 1).

In [ ]:
CONVENTIONS = [('Valid diagrams only', VO), ('Every run (failures scored 0)', ITT)]


def _anova_row(label, conv):
    an = A[f'anova_strategy__{conv}']
    term = an['terms'][an['term_of_interest']]
    summ = A[f'effect_sizes__{conv}']['summary']
    return {
        'Source': label,
        'df': f"{int(term['df'])}, {int(an['residual_df'])}",
        'F': f"{term['F']:.2f}",
        'p': p_apa(term['p']),
        'Partial eta^2': eta_apa(term['partial_eta_sq']),
        '|d| range': f"{summ['abs_d_min']:.2f} - {summ['abs_d_max']:.2f}",
    }


anova_f1 = pd.DataFrame(
    [_anova_row(label, conv) for label, conv in CONVENTIONS]
).set_index('Source')

# Name the widest contrast (identical under both conventions here).
_lc = A[f'effect_sizes__{ITT}']['summary']['largest_contrast']
print(f"Largest |d| contrast: {strat(_lc['group_a'])} vs {strat(_lc['group_b'])}")
anova_f1

## Two-way ANOVA of entity-identification F1

Strategy and input complexity (tier) as crossed factors, with the interaction
term testing whether complexity moderates the strategy effect. Intent-to-treat
convention, on 1200 per-cell means. Tier carries a large, highly significant
effect (harder inputs score lower); strategy and the interaction do not.

In [ ]:
anova_strategy_tier = anova_terms_table(f'anova_strategy_by_tier__{ITT}')
anova_strategy_tier

## Correctness and run outcomes by input complexity

Per-tier F1 under both scoring conventions, with the valid and failure rates.

The two F1 columns are **per-cell means**: repeats are averaged into one value
per strategy x model x input cell first, then those cells are averaged. This is
the convention `analysis.statistics` uses for every inferential number (see its
module docstring), so these figures match the ANOVA exactly. "F1 (every run)"
scores failed and invalid runs as zero; "F1 (valid only)" averages only the
cells' parsed runs.

The two rate columns are **pooled run fractions** of the tier total (a
different grain, appropriate for a count): valid rate is the parsed fraction,
fail rate the fraction that errored. They need not sum to 100 percent; the
remainder is diagrams that were returned but do not render.

In [ ]:
from decimal import ROUND_HALF_UP, Decimal

from maestro.viz import queries as q


def _pct(fraction):
    # Round half up so an exact .x5 lands up (0.55% -> 0.6%, 14.65% -> 14.7%),
    # matching the report. Snap to the counting grid first so float noise like
    # 0.5499999 does not defeat the rounding.
    pct = Decimal(str(fraction)).quantize(Decimal('0.0001')) * 100
    return f"{pct.quantize(Decimal('0.1'), ROUND_HALF_UP)}%"


# F1 columns use per-cell means (the analysis convention), so they match the
# ANOVA and the thesis exactly; the rate columns are pooled run fractions, a
# genuinely different grain.
f1_itt = per_cell_means(ITT).groupby('tier')['entity_id_f1'].mean()
f1_vo = per_cell_means(VO).groupby('tier')['entity_id_f1'].mean()
with viz_db.connect(DB_PATH) as conn:
    rate_rows = {t: (vr, fr) for t, n, vr, fr in q.run_rates_by_tier(conn)}

correctness_by_tier = pd.DataFrame(
    [{'Tier': t,
      'F1 (every run)': round(float(f1_itt[t]), 3),
      'F1 (valid only)': round(float(f1_vo[t]), 3),
      'Valid rate': _pct(rate_rows[t][0]),
      'Fail rate': _pct(rate_rows[t][1])}
     for t in sorted(rate_rows)]
).set_index('Tier')

correctness_by_tier

## Mean errors per valid diagram

Mean count of each error type per valid diagram, by strategy. Valid diagrams
only: an unparseable diagram scores every truth element as missing, which
would restate the reliability finding rather than describe error content.
Rates rather than sums, since each strategy has a different valid count. For
reference, a ground-truth diagram contains 15.6 entities and 17.3
relationships on average.

In [ ]:
from maestro.viz import queries as q

# Error types as rows, strategies as columns, in the report's order.
ERROR_ROWS = [
    ('missing_entities', 'Entities missing'),
    ('extra_entities', 'Entities extra'),
    ('false_entities', 'Entities false'),
    ('missing_relationships', 'Relationships missing'),
    ('extra_relationships', 'Relationships extra'),
    ('false_relationships', 'Relationships false'),
]
STRAT_COLS = [('single_agent', 'Single-agent'), ('sop_based', 'SOP-based'),
              ('crew_ai', 'CrewAI'), ('lang_graph', 'LangGraph')]

with viz_db.connect(DB_PATH) as conn:
    rates = q.taxonomy_rates_per_valid_diagram(
        conn, tuple(col for col, _ in ERROR_ROWS))

error_rates = pd.DataFrame(
    {label: {row_label: round(rates[s][col], 3) for col, row_label in ERROR_ROWS}
     for s, label in STRAT_COLS}
)
error_rates.index.name = 'Error Type'

error_rates

## Efficiency per run averaged over all runs

Mean tokens, cost, and latency per run, plus total cost, by strategy. Averaged
over every run including failures: a run that fails still consumes tokens and
is still paid for, so an efficiency figure that dropped them would understate
what a strategy costs. A per-run average is the natural grain for cost and
latency (unlike correctness, which uses per-cell means).

In [ ]:
from maestro.viz import queries as q

LABELS = {'single_agent': 'Single-agent', 'sop_based': 'SOP-based',
          'crew_ai': 'CrewAI', 'lang_graph': 'LangGraph'}
ORDER = ['single_agent', 'sop_based', 'crew_ai', 'lang_graph']

with viz_db.connect(DB_PATH) as conn:
    eff = {r[0]: r for r in q.efficiency_by_strategy(conn)}

efficiency = pd.DataFrame(
    [{'Strategy': LABELS[s],
      'Tokens': round(eff[s][1]),
      'Cost (USD)': round(eff[s][2], 4),
      'Latency (s)': round(eff[s][3], 1),
      'Total cost (USD)': round(eff[s][4], 2)}
     for s in ORDER]
).set_index('Strategy')

efficiency

## Two-way ANOVA of entity-identification F1 across models

Strategy and model as crossed factors, with the interaction testing whether
the strategy effect holds across models. Intent-to-treat convention, on 1200
per-cell means. Model carries a large, highly significant effect (model choice
drives correctness far more than orchestration); strategy and the interaction
do not.

In [ ]:
anova_strategy_model = anova_terms_table(f'anova_strategy_by_model__{ITT}')
anova_strategy_model

## Mixed-effects estimates for entity-identification F1

Fixed-effect coefficients from a linear mixed model fitted on the 6000
un-aggregated experimental runs, with model and input as crossed random
effects. Coefficients read against the single-agent baseline at tier 1; failed
and invalid runs are scored zero. Reported as a robustness check on the
aggregated ANOVA: the strategy contrasts stay non-significant while the tier
contrasts are large and negative, agreeing with the two-way ANOVA. Main effects
only; the intercept and the strategy x tier interaction terms are omitted.

In [ ]:
# Fixed-effect rows in report order: the three strategy contrasts against the
# single-agent baseline, then the two tier contrasts against tier 1. The
# intercept and the six strategy x tier interaction terms are omitted, as the
# table reports main effects only.
MIXED_ROWS = [
    ("C(strategy, Treatment(reference='single_agent'))[T.crew_ai]", 'CrewAI'),
    ("C(strategy, Treatment(reference='single_agent'))[T.sop_based]", 'SOP-based'),
    ("C(strategy, Treatment(reference='single_agent'))[T.lang_graph]", 'LangGraph'),
    ('C(tier)[T.2]', 'Tier 2'),
    ('C(tier)[T.3]', 'Tier 3'),
]

_fe = A[f'mixed_effects_robustness__{ITT}']['fixed_effects_estimates']

mixed_effects = pd.DataFrame(
    [{'Model': label,
      'Coefficient': f"{_fe[term]['coef']:+.3f}",
      'SE': round(_fe[term]['std_err'], 3),
      'p': p_apa(_fe[term]['p'])}
     for term, label in MIXED_ROWS]
).set_index('Model')

mixed_effects

## Reliability and accuracy per model

Per-model valid rate with F1 under both scoring conventions. F1 columns are
per-cell means (the analysis convention); valid rate is a pooled run fraction.
The printed correlations (over the ten models) contrast the two ways a model
can drive correctness: F1-every-run tracks the valid rate very closely (a model
is correct largely by producing renderable output), but barely tracks accuracy
on valid output (given a render, models score similarly). Failed and invalid
runs are scored zero in F1-every-run.

In [ ]:
import numpy as np
from scipy import stats as st

from maestro.viz import queries as q

# Models in the report's order (frontier then efficiency, grouped by vendor).
MODEL_ORDER = [
    'claude-opus-4-8', 'claude-haiku-4-5-20251001',
    'deepseek-v4-pro', 'deepseek-v4-flash',
    'gemini-3.5-flash', 'gemini-3.1-flash-lite',
    'mistral-medium-3-5', 'mistral-small-2603',
    'gpt-5.5-2026-04-23', 'gpt-5.4-mini-2026-03-17',
]

# F1 columns use per-cell means (the analysis convention); valid rate is a
# pooled run fraction, a different grain.
f1_itt = per_cell_means(ITT).groupby('model')['entity_id_f1'].mean()
f1_vo = per_cell_means(VO).groupby('model')['entity_id_f1'].mean()
with viz_db.connect(DB_PATH) as conn:
    valid = {m: vr for m, n, vr in q.valid_rate_by_model(conn)}

model_reliability = pd.DataFrame(
    [{'Model': m,
      'Valid rate': f'{valid[m] * 100:.1f}%',
      'F1 (valid only)': round(float(f1_vo[m]), 3),
      'F1 (every run)': round(float(f1_itt[m]), 3)}
     for m in MODEL_ORDER]
).set_index('Model')


def _corr_ci(a, b):
    r, _ = st.pearsonr(a, b)
    z, se = np.arctanh(r), 1 / np.sqrt(len(a) - 3)
    lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
    return r, lo, hi


_itt = np.array([f1_itt[m] for m in MODEL_ORDER])
_vo = np.array([f1_vo[m] for m in MODEL_ORDER])
_vr = np.array([valid[m] for m in MODEL_ORDER])


def _r(x):
    return f'{x:.3f}'.lstrip('0').replace('-0', '-')


r1, lo1, hi1 = _corr_ci(_itt, _vr)
r2, lo2, hi2 = _corr_ci(_itt, _vo)
print(f'Correctness vs valid rate:       r(8) = {_r(r1)}, '
      f'95% CI [{_r(lo1)}, {_r(hi1)}]')
print(f'Correctness vs accuracy on valid: r(8) = {_r(r2)}, '
      f'95% CI [{_r(lo2)}, {_r(hi2)}]')

model_reliability